# mnist 6

Full ciffer.

In [ ]:
from fastai.data.block import DataBlock, CategoryBlock
from fastai.data.external import untar_data, URLs
from fastai.data.transforms import get_image_files, parent_label, GrandparentSplitter
from fastai.metrics import error_rate
from fastai.vision.data import ImageBlock

path = untar_data(URLs.MNIST)
path.ls()

In [ ]:

datablock: DataBlock = DataBlock(
    blocks=[ImageBlock, CategoryBlock],
    get_items=get_image_files,
    get_y=parent_label,
    splitter=GrandparentSplitter(train_name='training', valid_name='testing')
)
dls = datablock.dataloaders(path, bs=256)
dls.show_batch()

In [ ]:
from torchvision.models import resnet34
from fastai.vision.learner import vision_learner

from fastai.callback.schedule import lr_find, minimum, steep, valley

learn = vision_learner(dls, resnet34, metrics=error_rate).to_fp16()
suggest_lr = learn.lr_find(suggest_funcs=(minimum, steep, valley))
suggest_lr

In [ ]:
from fastai.callback.schedule import fine_tune
learn.fine_tune(12, base_lr=abs(suggest_lr.steep - suggest_lr.valley)/2)

In [ ]:
from fastai.interpret import Interpretation

interp = Interpretation.from_learner(learn)
interp.plot_top_losses(9, figsize=(15,15))

In [ ]:
from fastai.interpret import ClassificationInterpretation

interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(12,12), dpi=60)

In [ ]:
from pathlib import Path

four = Path('/home/manu/courses/ai/fastbook/perso/tests')
images = get_image_files(four)
for image in images:
    pred = learn.predict(image)
    print(f'{image} == {pred[0]}')

In [ ]:
learn.export()